In [ ]:
import os
import random
import warnings
import math
from pathlib import Path

import numpy as np
import pandas as pd
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, precision_recall_curve, auc, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm

from opacus import PrivacyEngine
from opacus.accountants.utils import get_noise_multiplier

# ── Preparação do dataset ──────────────────────────────────────────────────────
df_original = pd.read_csv('csv/creditcard.csv')

df_classe_1 = df_original[df_original['Class'] == 1]
df_classe_0 = df_original[df_original['Class'] == 0]
df_classe_0_amostra = shuffle(df_classe_0, random_state=42).head(200000)

df_final = pd.concat([df_classe_1, df_classe_0_amostra])
df_final = shuffle(df_final, random_state=42).reset_index(drop=True)
df_final.to_csv('csv/credit.csv', index=False)

# ── Configurações globais ──────────────────────────────────────────────────────
warnings.filterwarnings("ignore", category=UserWarning)
os.environ["PYTHONUNBUFFERED"] = "1"

BASE_DIR   = Path.cwd()
CSV_DIR    = BASE_DIR / "csv"
MODELS_DIR = CSV_DIR / "models"
CSV_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

SEEDS      = [42]
ALPHAS     = [1]
EPSILONS   = [None, 1.0, 30.0]
ALGORITHMS = ["fedavg", "fedprox", "fedadam", "fedyogi", "fedadagrad"]

NUM_CLIENTS    = 3
NUM_ROUNDS     = 15
BATCH_SIZE     = 256
TARGET_DELTA   = 1e-5
MAX_GRAD_NORM  = 2.0

BASELINE_EPOCHS = 25
SHADOW_EPOCHS   = 20
VAL_FRACTION    = 0.20

def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

# Cache para get_noise_multiplier (chamada cara via accountant RDP)
DP_SIGMA_CACHE: dict = {}

def get_cached_dp_sigma(epsilon: float, sample_rate: float, total_epochs: int) -> float:
    key = (float(epsilon), round(float(sample_rate), 12), int(total_epochs), float(TARGET_DELTA))
    if key not in DP_SIGMA_CACHE:
        DP_SIGMA_CACHE[key] = float(
            get_noise_multiplier(
                target_epsilon=epsilon,
                target_delta=TARGET_DELTA,
                sample_rate=sample_rate,
                epochs=total_epochs,
                accountant="rdp",   # consistente com PrivacyEngine(accountant="rdp")
            )
        )
    return DP_SIGMA_CACHE[key]

In [ ]:
def make_stratify(y):
    unique, counts = np.unique(y, return_counts=True)
    return y if (len(unique) > 1 and counts.min() >= 2) else None

def safe_train_test_split(X, y, test_size, seed):
    strat = make_stratify(y)
    try:
        return train_test_split(X, y, test_size=test_size, random_state=seed, stratify=strat)
    except ValueError:
        return train_test_split(X, y, test_size=test_size, random_state=seed, stratify=None)

def generate_dirichlet_splits_for_seed(seed: int):
    seed_everything(seed)
    global_csv = CSV_DIR / "credit.csv"
    if not global_csv.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {global_csv}")

    df = pd.read_csv(global_csv)
    X = df.drop(columns=["Class"]).values
    y = df["Class"].values
    feature_cols = [c for c in df.columns if c != "Class"]

    # Split global: treino, validação e teste
    X_train_full, X_test_g, y_train_full, y_test_g = safe_train_test_split(X, y, test_size=0.20, seed=seed)
    X_train_g, X_val_g, y_train_g, y_val_g = safe_train_test_split(
        X_train_full, y_train_full, test_size=VAL_FRACTION, seed=seed
    )

    pd.DataFrame(np.c_[X_test_g, y_test_g], columns=feature_cols + ["Class"]).to_csv(
        CSV_DIR / f"credit_test_global_s{seed}.csv", index=False
    )
    pd.DataFrame(np.c_[X_val_g, y_val_g], columns=feature_cols + ["Class"]).to_csv(
        CSV_DIR / f"credit_val_global_s{seed}.csv", index=False
    )
    pd.DataFrame(np.c_[X_train_g, y_train_g], columns=feature_cols + ["Class"]).to_csv(
        CSV_DIR / f"credit_train_global_s{seed}.csv", index=False
    )

    # Dirichlet por alpha usando apenas o treino global
    df_train = pd.DataFrame(np.c_[X_train_g, y_train_g], columns=feature_cols + ["Class"])

    for alpha in ALPHAS:
        indices_bancos = {i: [] for i in range(NUM_CLIENTS)}

        for classe in [0, 1]:
            df_classe = df_train[df_train["Class"] == classe]
            indices = df_classe.index.values
            num_amostras = len(indices)

            proporcoes = np.random.dirichlet([alpha] * NUM_CLIENTS)
            contagem_por_banco = np.random.multinomial(num_amostras, proporcoes)

            start_idx = 0
            np.random.shuffle(indices)
            for i in range(NUM_CLIENTS):
                end_idx = start_idx + contagem_por_banco[i]
                indices_bancos[i].extend(indices[start_idx:end_idx])
                start_idx = end_idx

        for i in range(NUM_CLIENTS):
            df_banco = df_train.loc[indices_bancos[i]].copy()
            df_banco.to_csv(CSV_DIR / f"credit_banco_{i+1}_alpha_{alpha}_s{seed}.csv", index=False)

def build_data_bundle(alpha: int, seed: int):
    df_train = pd.read_csv(CSV_DIR / f"credit_train_global_s{seed}.csv")
    df_val = pd.read_csv(CSV_DIR / f"credit_val_global_s{seed}.csv")
    df_test = pd.read_csv(CSV_DIR / f"credit_test_global_s{seed}.csv")
    feature_cols = [c for c in df_train.columns if c != "Class"]
    
    X_train_raw = df_train[feature_cols].values.astype(np.float32)
    y_train_g = df_train["Class"].values.astype(np.float32)
    X_val_raw = df_val[feature_cols].values.astype(np.float32)
    y_val_g = df_val["Class"].values.astype(np.float32)
    X_test_raw = df_test[feature_cols].values.astype(np.float32)
    y_test_g = df_test["Class"].values.astype(np.float32)

    scaler = StandardScaler().fit(X_train_raw)
    X_train_global = scaler.transform(X_train_raw).astype(np.float32)
    X_val_global = scaler.transform(X_val_raw).astype(np.float32)
    X_test_global = scaler.transform(X_test_raw).astype(np.float32)

    pos = float(np.sum(y_train_g))
    neg = float(len(y_train_g) - pos)
    pos_weight_value = float(np.clip(neg / max(pos, 1.0), 1.0, 50.0))

    client_full_data = {}
    client_train_data = {}
    client_train_sizes = {}

    for client_id in range(1, NUM_CLIENTS + 1):
        df_c = pd.read_csv(CSV_DIR / f"credit_banco_{client_id}_alpha_{alpha}_s{seed}.csv")
        X_c_raw = df_c[feature_cols].values.astype(np.float32)
        y_c = df_c["Class"].values.astype(np.float32)

        X_c = scaler.transform(X_c_raw).astype(np.float32)
        client_full_data[client_id] = (X_c, y_c)

        if len(np.unique(y_c)) < 2 or np.sum(y_c) < 2:
            X_c_tr, y_c_tr = X_c, y_c
        else:
            X_c_tr, _, y_c_tr, _ = train_test_split(
                X_c,
                y_c,
                test_size=0.20,
                random_state=seed,
                stratify=make_stratify(y_c),
            )

        client_train_data[client_id] = (X_c_tr, y_c_tr)
        client_train_sizes[client_id] = len(y_c_tr)

    return {
        "feature_cols": feature_cols,
        "input_dim": X_train_raw.shape[1],
        "X_train_global": X_train_global,
        "X_val_global": X_val_global,
        "X_test_global": X_test_global,
        "y_train_global": y_train_g,
        "y_val_global": y_val_g,
        "y_test_global": y_test_g,
        "client_full_data": client_full_data,
        "client_train_data": client_train_data,
        "client_train_sizes": client_train_sizes,
        "pos_weight_value": pos_weight_value,
    }

In [ ]:
class FraudMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.out = nn.Linear(32, 1) 
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.out(x)

def get_model_parameters(model: nn.Module):
    return [val.detach().cpu().numpy() for _, val in model.state_dict().items()]

def set_model_parameters(model: nn.Module, parameters):
    state_dict = {k: torch.tensor(v, dtype=torch.float32) for k, v in zip(model.state_dict().keys(), parameters)}
    model.load_state_dict(state_dict, strict=True)

def clip_gradients_custom(model: nn.Module, max_norm: float = MAX_GRAD_NORM):
    # Usa MAX_GRAD_NORM (1.0) como default, igual ao DP — mesmas condições para comparações justas.
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

def _collect_probs(model: nn.Module, loader: DataLoader):
    model.eval()
    all_targets, all_probas = [], []

    with torch.no_grad():
        for inputs, targets in loader:
            probas = torch.sigmoid(model(inputs.float()).view(-1))
            all_targets.extend(targets.numpy().tolist())
            all_probas.extend(probas.numpy().tolist())

    return np.asarray(all_targets, dtype=np.float32), np.asarray(all_probas, dtype=np.float32)

def calibrate_threshold(y_true: np.ndarray, y_proba: np.ndarray):
    if len(np.unique(y_true)) < 2:
        return 0.5

    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    if len(thresholds) == 0:
        return 0.5

    precisions = precisions[:-1]
    recalls    = recalls[:-1]
    f1_scores  = 2.0 * (precisions * recalls) / (precisions + recalls + 1e-10)

    if len(f1_scores) == 0 or np.all(np.isnan(f1_scores)):
        return 0.5

    return float(thresholds[int(np.nanargmax(f1_scores))])

def evaluate_model(model: nn.Module, val_loader: DataLoader, test_loader: DataLoader, return_threshold=False):
    val_targets,  val_probas  = _collect_probs(model, val_loader)
    test_targets, test_probas = _collect_probs(model, test_loader)

    best_thresh = calibrate_threshold(val_targets, val_probas)
    test_preds  = (test_probas >= best_thresh).astype(np.float32)

    rec  = recall_score(test_targets, test_preds, zero_division=0)
    prec = precision_score(test_targets, test_preds, zero_division=0)

    if len(np.unique(test_targets)) < 2:
        auprc_val = float("nan")
    else:
        precision_pts, recall_pts, _ = precision_recall_curve(test_targets, test_probas)
        auprc_val = auc(recall_pts, precision_pts)

    if return_threshold:
        return auprc_val, rec, prec, best_thresh
    return auprc_val, rec, prec

In [ ]:
def weighted_average(client_params_list):
    total_examples = sum(n for _, n in client_params_list)
    if total_examples <= 0:
        raise ValueError("weighted_average recebeu total_examples <= 0.")

    num_layers = len(client_params_list[0][0])
    aggregated  = []

    for layer_idx in range(num_layers):
        layer_sum = None
        for params, n_examples in client_params_list:
            weighted  = np.asarray(params[layer_idx], dtype=np.float32) * (n_examples / total_examples)
            layer_sum = weighted if layer_sum is None else layer_sum + weighted
        aggregated.append(layer_sum)

    return aggregated

def get_algo_config(algo_name: str):
    configs = {
        "fedavg":     {"local_lr": 0.005, "local_epochs": 3, "optimizer": "adamw", "weight_decay": 0.0},
        "fedprox":    {"local_lr": 0.005, "local_epochs": 3, "optimizer": "adamw", "weight_decay": 0.0, "mu": 0.01},
        "fedadam":    {"local_lr": 0.005, "local_epochs": 3, "optimizer": "adamw", "weight_decay": 0.0, "server_lr": 0.02, "beta1": 0.9, "beta2": 0.99, "eps": 1e-8},
        "fedyogi":    {"local_lr": 0.005, "local_epochs": 3, "optimizer": "adamw", "weight_decay": 0.0, "server_lr": 0.02, "beta1": 0.9, "beta2": 0.99, "eps": 1e-8},
        "fedadagrad": {"local_lr": 0.005, "local_epochs": 3, "optimizer": "adamw", "weight_decay": 0.0, "server_lr": 0.05, "eps": 1e-8},
        "scaffold":   {"local_lr": 0.01,  "local_epochs": 3, "optimizer": "sgd",   "weight_decay": 0.0},
    }
    return configs[algo_name]

def fedadam_update(gp, ap, m, v, slr, b1, b2, eps):
    n_p, n_m, n_v = [], [], []
    for g, a, mi, vi in zip(gp, ap, m, v):
        d  = np.asarray(a) - np.asarray(g)
        mi = b1 * np.asarray(mi) + (1 - b1) * d
        vi = b2 * np.asarray(vi) + (1 - b2) * (d ** 2)
        n_p.append(np.asarray(g) + slr * mi / (np.sqrt(vi) + eps))
        n_m.append(mi); n_v.append(vi)
    return n_p, n_m, n_v

def fedyogi_update(gp, ap, m, v, slr, b1, b2, eps):
    n_p, n_m, n_v = [], [], []
    for g, a, mi, vi in zip(gp, ap, m, v):
        d  = np.asarray(a) - np.asarray(g)
        mi = b1 * np.asarray(mi) + (1 - b1) * d
        vi = np.asarray(vi) - (1 - b2) * np.sign(np.asarray(vi) - (d ** 2)) * (d ** 2)
        n_p.append(np.asarray(g) + slr * mi / (np.sqrt(np.maximum(vi, eps)) + eps))
        n_m.append(mi); n_v.append(vi)
    return n_p, n_m, n_v

def fedadagrad_update(gp, ap, v, slr, eps):
    n_p, n_v = [], []
    for g, a, vi in zip(gp, ap, v):
        d  = np.asarray(a) - np.asarray(g)
        vi = np.asarray(vi) + d ** 2
        n_p.append(np.asarray(g) + slr * d / (np.sqrt(vi) + eps))
        n_v.append(vi)
    return n_p, n_v

def scaffold_client_update(model, gp, sc, cc, lr, n_steps):
    """
    Atualiza os variates de controle do cliente (SCAFFOLD Algorithm 1, Option II).

    ATENÇÃO: o denominador deve ser `n_steps * lr`, onde n_steps é o número REAL
    de passos de otimizador (batches × épocas locais), NÃO o número de épocas.
    Usar épocas em vez de steps subestima o denominador em ceil(N/B) vezes,
    inflando as correções e causando divergência exponencial dos parâmetros.
    """
    denom = max(n_steps * lr, 1e-12)
    return [
        np.asarray(ci) - np.asarray(cs) + (np.asarray(wg) - np.asarray(wl)) / denom
        for wg, wl, cs, ci in zip(gp, get_model_parameters(model), sc, cc)
    ]

def scaffold_server_update(sc, cc_old, cc_new):
    return [
        np.asarray(s) + np.mean(
            [np.asarray(cc_new[cid][idx]) - np.asarray(cc_old[cid][idx]) for cid in cc_new],
            axis=0
        )
        for idx, s in enumerate(sc)
    ]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def eval_mia_metrics(y_mia_true, mia_scores, original_labels, filter_fraud=False):
    if filter_fraud:
        mask = (original_labels == 1)
        y_true_filtered = y_mia_true[mask]
        scores_filtered = mia_scores[mask]
    else:
        y_true_filtered = y_mia_true
        scores_filtered = mia_scores
        
    if len(np.unique(y_true_filtered)) < 2:
        return 0.50
        
    return roc_auc_score(y_true_filtered, scores_filtered)

# [NOVA FUNÇÃO] Substitui a get_confidence_scores para extrair 3 features (Loss, Proba, Classe)
def get_mia_features(model: nn.Module, X: np.ndarray, y: np.ndarray):
    model.eval()
    loader = DataLoader(
        TensorDataset(torch.tensor(X), torch.tensor(y)),
        batch_size=256, shuffle=False
    )
    features = []
    criterion = nn.BCEWithLogitsLoss(reduction='none')
    
    with torch.no_grad():
        for inputs, targets in loader:
            logits = model(inputs.float()).view(-1)
            losses = criterion(logits, targets.float()).cpu().numpy()
            probas = torch.sigmoid(logits).cpu().numpy()
            targets_np = targets.numpy()
            
            for l, p, t in zip(losses, probas, targets_np):
                features.append([l, p, t])
                
    return np.asarray(features, dtype=np.float32)

def run_mia_evaluations(
    model, X_train, y_train, X_test, y_test,
    input_dim, pos_weight_value, seed, shadow_epochs=SHADOW_EPOCHS
):
    # ── 1) Black-box (Ataque Métrica Simples) ─────────────────────────────────
    feat_train = get_mia_features(model, X_train, y_train)
    feat_test  = get_mia_features(model, X_test,  y_test)

    # Usa a probabilidade (coluna 1) para calcular a confiança no estilo antigo
    conf_train = np.where(y_train == 1, feat_train[:, 1], 1.0 - feat_train[:, 1])
    conf_test  = np.where(y_test == 1, feat_test[:, 1], 1.0 - feat_test[:, 1])

    scores_bb  = np.concatenate([conf_train, conf_test]).flatten()
    y_mia_true = np.concatenate([np.ones(len(conf_train)), np.zeros(len(conf_test))])
    original_labels = np.concatenate([y_train, y_test])

    auc_bb_all   = eval_mia_metrics(y_mia_true, scores_bb, original_labels, filter_fraud=False)
    auc_bb_fraud = eval_mia_metrics(y_mia_true, scores_bb, original_labels, filter_fraud=True)

    # ── 2) Shadow model (Ataque ML com Random Forest) ─────────────────────────
    if len(np.unique(y_train)) < 2:
        return auc_bb_all, auc_bb_fraud, 0.50, 0.50

    Xs_in, Xs_out, ys_in, ys_out = train_test_split(
        X_train, y_train,
        test_size=0.5, random_state=seed, stratify=make_stratify(y_train)
    )

    shadow_model = train_model_standard(
        Xs_in, ys_in, input_dim, pos_weight_value,
        epochs=shadow_epochs, seed=seed
    )

    feat_shadow_in  = get_mia_features(shadow_model, Xs_in,  ys_in)
    feat_shadow_out = get_mia_features(shadow_model, Xs_out, ys_out)

    X_attack_tr = np.vstack([feat_shadow_in, feat_shadow_out])
    y_attack_tr = np.concatenate([np.ones(len(feat_shadow_in)), np.zeros(len(feat_shadow_out))])

    # Classificador muito mais potente que a Regressão Logística
    attack_classifier = RandomForestClassifier(
        n_estimators=100, max_depth=5, class_weight="balanced", random_state=seed
    )
    attack_classifier.fit(X_attack_tr, y_attack_tr)

    # Avaliar no modelo ALVO
    feat_target_in  = get_mia_features(model, X_train, y_train)
    feat_target_out = get_mia_features(model, X_test,  y_test)
    X_attack_te  = np.vstack([feat_target_in, feat_target_out])

    scores_sh = attack_classifier.predict_proba(X_attack_te)[:, 1]

    auc_sh_all   = eval_mia_metrics(y_mia_true, scores_sh, original_labels, filter_fraud=False)
    auc_sh_fraud = eval_mia_metrics(y_mia_true, scores_sh, original_labels, filter_fraud=True)

    return auc_bb_all, auc_bb_fraud, auc_sh_all, auc_sh_fraud

In [ ]:
def train_model_standard(X_train, y_train, input_dim, pos_weight_value, epochs=40, lr=0.005, seed=42):
    """Treina os baselines centralizados e locais sem privacidade — usa TODOS os dados recebidos."""
    seed_everything(seed)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    model     = FraudMLP(input_dim)
    # [ALTERADO] weight_decay zerado
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0) 
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32))

    model.train()
    for _ in range(epochs):
        for inputs, targets in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(inputs.float()).view(-1), targets.float().view(-1))
            loss.backward()
            clip_gradients_custom(model)
            optimizer.step()
    return model


def train_client_round(
    client_id,
    global_params,
    client_X_train,
    client_y_train,
    input_dim,
    pos_weight_value,
    algo_name,
    algo_cfg,
    epsilon,
    seed,
    server_c=None,
    client_c=None,
    dp_sigma=None,
    scaffold_state=None,   # dict mutável {"sc": [...], "cc": [...]} compartilhado com o hook DP
):
    loader = DataLoader(
        TensorDataset(torch.from_numpy(client_X_train), torch.from_numpy(client_y_train)),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    model = FraudMLP(input_dim=input_dim)
    set_model_parameters(model, global_params)

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32))
    optimizer = (
        optim.SGD(model.parameters(), lr=algo_cfg["local_lr"], momentum=0.0, weight_decay=algo_cfg["weight_decay"])
        if algo_name == "scaffold"
        else optim.AdamW(model.parameters(), lr=algo_cfg["local_lr"], weight_decay=algo_cfg["weight_decay"])
    )

    if epsilon is not None:
        # FIX accountant: PrivacyEngine deve usar o mesmo accountant (rdp) que get_noise_multiplier,
        # caso contrário get_epsilon() pode retornar NaN ou valores inconsistentes.
        privacy_engine = PrivacyEngine(accountant="rdp")
        model, optimizer, loader = privacy_engine.make_private(
            module=model,
            optimizer=optimizer,
            data_loader=loader,
            noise_multiplier=dp_sigma,
            max_grad_norm=MAX_GRAD_NORM,
        )

        if algo_name == "scaffold":
            accounting_hook = optimizer.step_hook

            def _scaffold_dp_hook(dp_opt):
                # Preserva o accounting de privacidade registrado pelo make_private
                if accounting_hook is not None:
                    accounting_hook(dp_opt)
                # Aplica a correção do SCAFFOLD ao p.grad já processado pelo DP
                with torch.no_grad():
                    unwrapped = model._module if hasattr(model, "_module") else model
                    for p, sc_i, cc_i in zip(unwrapped.parameters(), scaffold_state["sc"], scaffold_state["cc"]):
                        if p.grad is not None:
                            p.grad.add_(sc_i - cc_i)

            optimizer.attach_step_hook(_scaffold_dp_hook)

    sc_tensors = [torch.tensor(c, dtype=torch.float32) for c in server_c] if algo_name == "scaffold" else None
    cc_tensors = [torch.tensor(c, dtype=torch.float32) for c in client_c] if algo_name == "scaffold" else None

    if algo_name == "scaffold" and epsilon is not None:
        # Atualiza o container mutável lido pelo hook — feito uma vez antes das épocas locais,
        # pois sc/cc são fixos durante o treino local de uma rodada.
        scaffold_state["sc"] = sc_tensors
        scaffold_state["cc"] = cc_tensors

    model.train()
    for _ in range(algo_cfg["local_epochs"]):
        for inputs, targets in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(inputs.float()).view(-1), targets.float().view(-1))

            if algo_name == "fedprox":
                unwrapped = model._module if hasattr(model, "_module") else model
                prox_term = sum(
                    torch.sum((p - torch.tensor(gp, dtype=torch.float32, device=p.device)) ** 2)
                    for p, gp in zip(unwrapped.parameters(), global_params)
                )
                loss = loss + (algo_cfg["mu"] / 2.0) * prox_term

            loss.backward()

            # SCAFFOLD sem DP: correção direta em p.grad (seguro pois não há hook do Opacus)
            if algo_name == "scaffold" and epsilon is None:
                with torch.no_grad():
                    for p, sc_i, cc_i in zip(model.parameters(), sc_tensors, cc_tensors):
                        if p.grad is not None:
                            p.grad.add_(sc_i - cc_i)

            if epsilon is None:
                clip_gradients_custom(model)

            optimizer.step()

    model_unwrapped = model._module if hasattr(model, "_module") else model
    new_params = get_model_parameters(model_unwrapped)

    # FIX scaffold_client_update: usa n_steps (passos reais de otimizador = batches × épocas),
    # NÃO o número de épocas. Usar épocas subestima o denominador em ~ceil(N/B) vezes e
    # causa divergência exponencial dos variates de controle.
    n_steps = math.ceil(len(loader.dataset) / BATCH_SIZE) * algo_cfg["local_epochs"]
    new_client_c = (
        scaffold_client_update(model_unwrapped, global_params, server_c, client_c, algo_cfg["local_lr"], n_steps)
        if algo_name == "scaffold"
        else None
    )
    return new_params, len(loader.dataset), new_client_c


def run_fl_experiment(algo_name, bundle, epsilon, seed):
    """Executa as rodadas globais do aprendizado federado."""
    algo_cfg     = get_algo_config(algo_name)
    global_model = FraudMLP(input_dim=bundle["input_dim"])
    global_params = get_model_parameters(global_model)

    server_c, client_cs, m, v = None, None, None, None
    if algo_name == "scaffold":
        server_c   = [np.zeros_like(p, dtype=np.float32) for p in global_params]
        client_cs  = {cid: [np.zeros_like(p, dtype=np.float32) for p in global_params] for cid in range(1, NUM_CLIENTS + 1)}
    elif algo_name in ["fedadam", "fedyogi"]:
        m = [np.zeros_like(p, dtype=np.float32) for p in global_params]
        v = [np.zeros_like(p, dtype=np.float32) for p in global_params]
    elif algo_name == "fedadagrad":
        v = [np.zeros_like(p, dtype=np.float32) for p in global_params]

    # FIX epsilon: sigma deve ser calibrado para o total de épocas locais de TODAS as rodadas.
    # Calibrar para apenas local_epochs (4) e reutilizar por NUM_ROUNDS=20 rodadas consome
    # ~4–4.3× mais epsilon do que o declarado (ex.: ε=1 real ≈ 4.65, ε=5 real ≈ 21.45).
    total_local_epochs = NUM_ROUNDS * algo_cfg["local_epochs"]
    dp_sigma_by_client = None
    if epsilon is not None:
        dp_sigma_by_client = {
            cid: get_cached_dp_sigma(
                epsilon=epsilon,
                sample_rate=min(1.0, BATCH_SIZE / max(bundle["client_train_sizes"][cid], 1)),
                total_epochs=total_local_epochs,
            )
            for cid in range(1, NUM_CLIENTS + 1)
        }

    # Um scaffold_state por cliente: container mutável lido pelo hook DP do SCAFFOLD
    scaffold_states = (
        {cid: {"sc": None, "cc": None} for cid in range(1, NUM_CLIENTS + 1)}
        if algo_name == "scaffold" else {}
    )

    for round_num in range(1, NUM_ROUNDS + 1):
        client_updates = []
        scaffold_c_new = {}

        for client_id in range(1, NUM_CLIENTS + 1):
            X_c, y_c = bundle["client_train_data"][client_id]
            params, num_ex, new_c = train_client_round(
                client_id,
                global_params,
                X_c,
                y_c,
                bundle["input_dim"],
                bundle["pos_weight_value"],
                algo_name,
                algo_cfg,
                epsilon,
                seed,
                server_c,
                client_cs[client_id] if client_cs else None,
                None if dp_sigma_by_client is None else dp_sigma_by_client[client_id],
                scaffold_state=scaffold_states.get(client_id),
            )
            client_updates.append((params, num_ex))
            if algo_name == "scaffold":
                scaffold_c_new[client_id] = new_c

        avg_params = weighted_average(client_updates)

        if algo_name in ["fedavg", "fedprox", "scaffold"]:
            global_params = avg_params
        elif algo_name == "fedadam":
            global_params, m, v = fedadam_update(global_params, avg_params, m, v, algo_cfg["server_lr"], algo_cfg["beta1"], algo_cfg["beta2"], algo_cfg["eps"])
        elif algo_name == "fedyogi":
            global_params, m, v = fedyogi_update(global_params, avg_params, m, v, algo_cfg["server_lr"], algo_cfg["beta1"], algo_cfg["beta2"], algo_cfg["eps"])
        elif algo_name == "fedadagrad":
            global_params, v = fedadagrad_update(global_params, avg_params, v, algo_cfg["server_lr"], algo_cfg["eps"])

        if algo_name == "scaffold":
            server_c  = scaffold_server_update(server_c, client_cs, scaffold_c_new)
            client_cs = scaffold_c_new

    set_model_parameters(global_model, global_params)
    return global_model

In [ ]:
def log_step(message):
    """Função auxiliar para printar logs sem quebrar a barra do tqdm"""
    timestamp = datetime.now().strftime("%H:%M:%S")
    tqdm.write(f"[{timestamp}] {message}")

def _safe_label(value):
    if value is None:
        return "Sem_DP"
    if isinstance(value, float) and float(value).is_integer():
        return str(int(value))
    return str(value).replace(".", "p").replace("/", "_")

def _artifact_path(strategy, seed, alpha_label, epsilon_label, client_id=None):
    parts = [
        strategy,
        f"s{seed}",
        f"a{_safe_label(alpha_label)}",
        f"e{_safe_label(epsilon_label)}",
    ]
    if client_id is not None:
        parts.append(f"c{client_id}")
    fname = "__".join(parts) + ".pt"
    return MODELS_DIR / fname

def run_all_experiments():
    raw_results = []
    manifest_rows = []

    log_step("=== INICIANDO PIPELINE DE TREINAMENTO FEDERADO E BASELINES ===")

    for seed in tqdm(SEEDS, desc="Processando Seeds", position=0):
        log_step(f"\n{'='*50}\n[!] INICIANDO SEED {seed}\n{'='*50}")
        
        log_step(f"Gerando splits de Dirichlet para Seed {seed}...")
        generate_dirichlet_splits_for_seed(seed)

        bundle_seed = build_data_bundle(ALPHAS[0], seed)
        val_loader = DataLoader(
            TensorDataset(
                torch.tensor(bundle_seed["X_val_global"]),
                torch.tensor(bundle_seed["y_val_global"])
            ),
            batch_size=256,
            shuffle=False
        )
        test_loader = DataLoader(
            TensorDataset(
                torch.tensor(bundle_seed["X_test_global"]),
                torch.tensor(bundle_seed["y_test_global"])
            ),
            batch_size=256,
            shuffle=False
        )

        # ===== BASELINE CENTRAL =====
        log_step(f"Treinando Baseline Central (Seed {seed})...")
        model_central = train_model_standard(
            bundle_seed["X_train_global"],
            bundle_seed["y_train_global"],
            bundle_seed["input_dim"],
            bundle_seed["pos_weight_value"],
            epochs=BASELINE_EPOCHS,
            seed=seed
        )
        aup_c, rec_c, pre_c, th_c = evaluate_model(model_central, val_loader, test_loader, return_threshold=True)

        central_path = _artifact_path("Baseline_Central", seed, "Global", "Sem_DP")
        torch.save(model_central.state_dict(), central_path)

        raw_results.append({
            "Seed": seed,
            "AlphaLabel": "Global",
            "AlphaSort": -1,
            "EpsilonLabel": "Sem_DP",
            "EpsilonSort": -1,
            "Strategy": "Baseline Central",
            "AUPRC": aup_c,
            "Recall": rec_c,
            "Prec": pre_c,
            "Threshold": th_c,
        })
        manifest_rows.append({
            "Seed": seed,
            "AlphaLabel": "Global",
            "AlphaSort": -1,
            "EpsilonLabel": "Sem_DP",
            "EpsilonSort": -1,
            "Strategy": "Baseline Central",
            "TrainScope": "global",
            "ClientID": "",
            "ModelPath": str(central_path),
        })
        log_step(f"Baseline Central concluído. AUPRC: {aup_c:.4f}")

        # ===== LOOP FEDERADO =====
        for alpha in tqdm(ALPHAS, desc=f"Alphas (Seed {seed})", position=1, leave=False):
            log_step(f"--- Configurando dados para Alpha = {alpha} ---")
            bundle = build_data_bundle(alpha, seed)

            for epsilon in tqdm(EPSILONS, desc=f"Epsilons (Alpha {alpha})", position=2, leave=False):
                eps_label = "Sem_DP" if epsilon is None else str(epsilon)
                eps_sort = -1 if epsilon is None else float(epsilon)

                if epsilon is None:
                    log_step(f"Treinando Baselines Locais (Alpha {alpha}, Sem DP)...")
                    for cid in range(1, NUM_CLIENTS + 1):
                        X_c, y_c = bundle["client_full_data"][cid]
                        model_l = train_model_standard(
                            X_c,
                            y_c,
                            bundle["input_dim"],
                            bundle["pos_weight_value"],
                            seed=seed,
                        )
                        
                        aup_l, rec_l, pre_l, th_l = evaluate_model(model_l, val_loader, test_loader, return_threshold=True)
                        
                        # Salvar modelo local e colocar no manifesto
                        local_path = _artifact_path(f"Local_Silo_{cid}", seed, alpha, "Sem_DP", client_id=cid)
                        torch.save(model_l.state_dict(), local_path)
                        
                        auc_bb_la, auc_bb_lf, auc_sh_la, auc_sh_lf = run_mia_evaluations(
                            model_l, X_c, y_c,
                            bundle["X_test_global"], bundle["y_test_global"],
                            bundle["input_dim"], bundle["pos_weight_value"], seed,
                        )
                        
                        raw_results.append({
                            "Seed": seed,
                            "AlphaLabel": str(alpha),
                            "AlphaSort": float(alpha),
                            "EpsilonLabel": eps_label,
                            "EpsilonSort": eps_sort,
                            "Strategy": f"Local_Silo_{cid}",
                            "AUPRC": aup_l,
                            "Recall": rec_l,
                            "Prec": pre_l,
                            "Threshold": th_l,
                            "BB_All": auc_bb_la,
                            "BB_Fraud": auc_bb_lf,
                            "Sh_All": auc_sh_la,
                            "Sh_Fraud": auc_sh_lf,
                        })
                        manifest_rows.append({
                            "Seed": seed,
                            "AlphaLabel": str(alpha),
                            "AlphaSort": float(alpha),
                            "EpsilonLabel": eps_label,
                            "EpsilonSort": eps_sort,
                            "Strategy": f"Local_Silo_{cid}",
                            "TrainScope": "client",
                            "ClientID": cid,
                            "ModelPath": str(local_path),
                        })

                for algo in tqdm(ALGORITHMS, desc=f"Algoritmos FL (Eps {eps_label})", position=3, leave=False):
                    log_step(f"Iniciando {algo.upper()} [Alpha: {alpha} | DP(ε): {eps_label}]")
                    model_f = run_fl_experiment(algo, bundle, epsilon, seed)
                    aup_f, rec_f, pre_f, th_f = evaluate_model(model_f, val_loader, test_loader, return_threshold=True)

                    fl_path = _artifact_path(f"FL_{algo.upper()}", seed, alpha, eps_label)
                    torch.save(model_f.state_dict(), fl_path)

                    raw_results.append({
                        "Seed": seed,
                        "AlphaLabel": str(alpha),
                        "AlphaSort": float(alpha),
                        "EpsilonLabel": eps_label,
                        "EpsilonSort": eps_sort,
                        "Strategy": f"FL_{algo.upper()}",
                        "AUPRC": aup_f,
                        "Recall": rec_f,
                        "Prec": pre_f,
                        "Threshold": th_f,
                    })
                    manifest_rows.append({
                        "Seed": seed,
                        "AlphaLabel": str(alpha),
                        "AlphaSort": float(alpha),
                        "EpsilonLabel": eps_label,
                        "EpsilonSort": eps_sort,
                        "Strategy": f"FL_{algo.upper()}",
                        "TrainScope": "global",
                        "ClientID": "",
                        "ModelPath": str(fl_path),
                    })
                    log_step(f"Concluído {algo.upper()}. AUPRC: {aup_f:.4f}")

        df_seed = pd.DataFrame([r for r in raw_results if r["Seed"] == seed])
        completa_seed = df_seed.sort_values(["AUPRC", "Recall", "Prec"], ascending=False)[
            ["Strategy", "AlphaLabel", "EpsilonLabel", "AUPRC", "Recall", "Prec", "Threshold"]
        ]
        
        log_step(f"\n--- TABELA COMPLETA DE RESULTADOS: SEED {seed} ---")
        tqdm.write(completa_seed.to_markdown(tablefmt="pipe", index=False))
        log_step(f"Seed {seed} finalizada com sucesso.\n")

    log_step("Agregando resultados finais de todas as seeds...")
    df_raw = pd.DataFrame(raw_results)
    df_raw.to_csv(CSV_DIR / "log_todas_seeds_bruto.csv", index=False)

    df_manifest = pd.DataFrame(manifest_rows)
    df_manifest.to_csv(CSV_DIR / "manifest_modelos.csv", index=False)

    cols_metrics = ["AUPRC", "Recall", "Prec"]
    df_grouped = (
        df_raw
        .groupby(["Strategy", "AlphaLabel", "AlphaSort", "EpsilonLabel", "EpsilonSort"], dropna=False)[cols_metrics]
        .agg(["mean", "std"])
        .reset_index()
    )

    def format_mean_std(row, col):
        mean = row[(col, "mean")]
        std = row[(col, "std")]
        if pd.isna(std) or std == 0.0:
            return f"{mean:.4f}"
        return f"{mean:.4f} ± {std:.4f}"

    df_final = pd.DataFrame({
        "Estratégia": df_grouped["Strategy"],
        "Alpha": df_grouped["AlphaLabel"],
        "DP(ε)": df_grouped["EpsilonLabel"],
    })

    for col in cols_metrics:
        df_final[col] = df_grouped.apply(lambda r: format_mean_std(r, col), axis=1)

    df_final["AlphaSort"] = df_grouped["AlphaSort"].values
    df_final["EpsilonSort"] = df_grouped["EpsilonSort"].values
    df_final.sort_values(by=["AlphaSort", "EpsilonSort", "Estratégia"], inplace=True)
    df_final.drop(columns=["AlphaSort", "EpsilonSort"], inplace=True)

    log_step("=" * 100)
    log_step("RESULTADOS FINAIS ROBUSTOS (MÉDIA ± DESVIO PADRÃO DAS SEEDS)")
    log_step("=" * 100)
    tqdm.write(df_final.to_markdown(tablefmt="pipe", index=False))
    log_step("=" * 100)

    df_final.to_csv(CSV_DIR / "resultados_finais_agregados.csv", index=False)
    log_step("Processo de Treinamento totalmente finalizado!")
    return df_final, df_manifest

if __name__ == "__main__":
    run_all_experiments()

In [ ]:
def load_model_from_path(model_path: str, input_dim: int):
    model = FraudMLP(input_dim)
    state_dict = torch.load(model_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model

def run_all_mia_from_manifest():
    manifest_path = CSV_DIR / "manifest_modelos.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest não encontrado: {manifest_path}")

    log_step("=== INICIANDO AVALIAÇÕES DE ATAQUE MIA ===")
    df_manifest = pd.read_csv(manifest_path)
    mia_rows = []
    bundle_cache = {}

    for _, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc="Executando MIA"):
        seed = int(row["Seed"])
        alpha_sort = int(row["AlphaSort"])
        alpha_key = ALPHAS[0] if alpha_sort < 0 else int(alpha_sort)
        strategy = row["Strategy"]

        log_step(f"MIA -> Avaliando: {strategy} (Seed {seed} | Alpha {alpha_key})")

        cache_key = (seed, alpha_key)
        if cache_key not in bundle_cache:
            bundle_cache[cache_key] = build_data_bundle(alpha_key, seed)

        bundle = bundle_cache[cache_key]
        model = load_model_from_path(row["ModelPath"], bundle["input_dim"])

        if row["TrainScope"] == "client":
            cid = int(row["ClientID"])
            X_train_mia, y_train_mia = bundle["client_full_data"][cid]
        else:
            X_train_mia = bundle["X_train_global"]
            y_train_mia = bundle["y_train_global"]

        X_test_mia = bundle["X_test_global"]
        y_test_mia = bundle["y_test_global"]

        bb_all, bb_fraud, sh_all, sh_fraud = run_mia_evaluations(
            model=model,
            X_train=X_train_mia,
            y_train=y_train_mia,
            X_test=X_test_mia,
            y_test=y_test_mia,
            input_dim=bundle["input_dim"],
            pos_weight_value=bundle["pos_weight_value"],
            seed=seed,
            shadow_epochs=SHADOW_EPOCHS
        )

        mia_rows.append({
            "Seed": seed,
            "AlphaLabel": row["AlphaLabel"],
            "AlphaSort": float(row["AlphaSort"]),
            "EpsilonLabel": row["EpsilonLabel"],
            "EpsilonSort": float(row["EpsilonSort"]),
            "Strategy": strategy,
            "BB_All": bb_all,
            "BB_Fraud": bb_fraud,
            "Sh_All": sh_all,
            "Sh_Fraud": sh_fraud,
        })

    log_step("Agregando resultados do MIA...")
    df_mia = pd.DataFrame(mia_rows)
    df_mia.to_csv(CSV_DIR / "log_mia_bruto.csv", index=False)

    cols_metrics = ["BB_All", "BB_Fraud", "Sh_All", "Sh_Fraud"]
    df_grouped = (
        df_mia
        .groupby(["Strategy", "AlphaLabel", "AlphaSort", "EpsilonLabel", "EpsilonSort"], dropna=False)[cols_metrics]
        .agg(["mean", "std"])
        .reset_index()
    )

    def format_mean_std(row, col):
        mean = row[(col, "mean")]
        std = row[(col, "std")]
        if pd.isna(std) or std == 0.0:
            return f"{mean:.4f}"
        return f"{mean:.4f} ± {std:.4f}"

    df_final = pd.DataFrame({
        "Estratégia": df_grouped["Strategy"],
        "Alpha": df_grouped["AlphaLabel"],
        "DP(ε)": df_grouped["EpsilonLabel"],
    })

    for col in cols_metrics:
        df_final[col] = df_grouped.apply(lambda r: format_mean_std(r, col), axis=1)

    df_final["AlphaSort"] = df_grouped["AlphaSort"].values
    df_final["EpsilonSort"] = df_grouped["EpsilonSort"].values
    df_final.sort_values(by=["AlphaSort", "EpsilonSort", "Estratégia"], inplace=True)
    df_final.drop(columns=["AlphaSort", "EpsilonSort"], inplace=True)

    log_step("=" * 100)
    log_step("RESULTADOS FINAIS DO MIA (MÉDIA ± DESVIO PADRÃO DAS SEEDS)")
    log_step("=" * 100)
    tqdm.write(df_final.to_markdown(tablefmt="pipe", index=False))
    log_step("=" * 100)

    df_final.to_csv(CSV_DIR / "resultados_mia_agregados.csv", index=False)
    log_step("Ataques MIA totalmente finalizados!")
    return df_final

if __name__ == "__main__":
    run_all_mia_from_manifest()